# Оптимизация waveform: пример для перехода GS0→GS3

End-to-end pipeline по главе 5 курсовой:
1. Загрузить калиброванные параметры (`data/calibration/fitted_params.json` из M3).
2. Построить `PixelOdeSim` и обучить `SurrogateModel`.
3. Решить одну PMP-задачу из § 5.2 (`solve_pmp_slice`).
4. Сравнить найденный waveform с типичным заводским B0.
5. Сохранить рисунок в `tex/figures/plots/optimal_waveform_example.pdf` (используется в § 6.1).

Реализация — в `scripts/run_optimize.py` (notebook — тонкий wrapper, как у `01_calibration.ipynb`).

In [3]:
import json
import sys
from pathlib import Path

sys.path.insert(0, '..')
from scripts.run_optimize import main

result = main()
print(json.dumps(result, indent=2, ensure_ascii=False))

INFO python.surrogate: fit_from_ode_sim: grid 5V × 6T × 4Z = 120 points
INFO python.surrogate: fit complete: e_grid mean=0.0042 мДж, dz_grid mean=0.0079
INFO python.optimizer: solve_pmp_slice: z_init=0.070, z_target=0.400, ε_G=0.0400, ε_τ=0.400s, K_eff_max=4
INFO python.optimizer: best: E=0.0117 мДж, G=0.0024, τ=0.320 с (K_eff=4) | total=432750, eval=3188
INFO scripts.run_optimize: solved: E=0.012 мДж, G=0.0024, τ=0.320 с
INFO scripts.run_optimize: saved: /Users/georgii/GITHUB/MAGCOURSEWORK/tex/figures/plots/optimal_waveform_example.pdf


{
  "ok": true,
  "voltages": [
    -15.0,
    -5.0,
    5.0,
    15.0
  ],
  "durations_frames": [
    4,
    4,
    4,
    4
  ],
  "energy_mJ": 0.011733333333333332,
  "ghost_residual": 0.0024375000000002034,
  "latency_s": 0.32,
  "k_eff": 4,
  "charge_imbalance_Vs": -2.220446049250313e-16
}


## Интерпретация результата

Найденный waveform состоит из K_eff = 4 активных фаз — ровно граница теоремы 4.1 (`K_eff ≤ d_state + 2 = 4`). Перестановка `(−15, −5, +5, +15)` с одинаковой длительностью каждой фазы даёт идеальный заряд-балaнс `Σ V·T = 0` (`charge_imbalance_Vs = 0`), что согласуется с § 4.3.

Энергия на пиксель ≈ 12 мкДж — это согласуется с формулой Lin 2024 при `C_eff = 11.7 нФ` и совокупной длительности `τ = 320 мс`. Для оценки на всю панель (250×122 пикселя) умножается на число пикселей.

Графическое сопоставление с заводским B0 сохранено в `tex/figures/plots/optimal_waveform_example.pdf` и встраивается в § 6.1 курсовой.

In [4]:
# Отдельная визуализация найденного waveform в notebook (для интерактивной проверки)
import matplotlib.pyplot as plt
from PIL import Image

pdf_path = Path('../tex/figures/plots/optimal_waveform_example.pdf')
if pdf_path.exists():
    print(f'PDF сохранён: {pdf_path} ({pdf_path.stat().st_size} байт)')
else:
    print('PDF не найден — выполните ячейку выше.')

PDF сохранён: ../tex/figures/plots/optimal_waveform_example.pdf (23056 байт)
